# Harta identity pilot and six-source calibration — 2026-10-22
Executed, read-only evidence checks. Private captures/reports must be present.
Forty primary pairs are not a representative truth-labelled duplicate sample.
A retained nonprimary pair is excluded from the primary denominator.


In [1]:
import json
from pathlib import Path
from collections import Counter
root=Path('reports/generated/harta/matching_pilot')
pool=json.loads(Path('.tmp/harta-matching-pilot/candidate_pool_private.json').read_text())
reviews=json.loads((root/'pair_reviews_private.json').read_text())
assert sum(r['source_website']=='harta' for r in pool)==50
assert len(reviews)==len({r['harta_source_id'] for r in reviews})==40
print('Source pool counts:', dict(Counter(r['source_website'] for r in pool)))
print('Primary reviewed pairs:',len(reviews))
print('Reference portals:',dict(Counter(r['other_source'] for r in reviews)))

Source pool counts: {'gjirafa': 4330, 'harta': 50, 'merrjep': 14862, 'myrealestate': 136, 'pro-rks': 3434, 'topia': 468, 'vision': 102}
Primary reviewed pairs: 40
Reference portals: {'gjirafa': 5, 'merrjep': 12, 'pro-rks': 13, 'vision': 3, 'topia': 4, 'myrealestate': 3}


In [2]:
from groundtruth.analytics.harta_matching import matching_gate
gate=matching_gate(reviews)
print('Identity verdicts:',{k:gate[k] for k in ['verified_true','verified_false','unresolved']})
print('Precision:',gate['precision_among_resolved'])
print('False-match rate:',gate['false_match_rate_among_resolved'])
assert gate['unresolved']==40 and gate['precision_among_resolved'] is None
assert not gate['research_matching_support_enabled']
assert not gate['automatic_merge_enabled'] and not gate['field_transfer_enabled']
for r in reviews:
    print(r['pair_id'],r['other_source'],r['verdict'], 'unit keys:',bool(r['harta_facts']['apartment_unit_keys']),bool(r['other_facts']['apartment_unit_keys']), 'cross-reference:',r['direct_listing_cross_reference'])

Identity verdicts: {'verified_true': 0, 'verified_false': 0, 'unresolved': 40}
Precision: None
False-match rate: None
HCM-001 gjirafa unresolved unit keys: False False cross-reference: False
HCM-002 gjirafa unresolved unit keys: False False cross-reference: False
HCM-003 gjirafa unresolved unit keys: False False cross-reference: False
HCM-004 merrjep unresolved unit keys: False False cross-reference: False
HCM-005 merrjep unresolved unit keys: False False cross-reference: False
HCM-006 merrjep unresolved unit keys: False False cross-reference: False
HCM-007 pro-rks unresolved unit keys: False False cross-reference: False
HCM-008 pro-rks unresolved unit keys: False False cross-reference: False
HCM-009 pro-rks unresolved unit keys: False False cross-reference: False
HCM-010 vision unresolved unit keys: False False cross-reference: False
HCM-011 vision unresolved unit keys: False False cross-reference: False
HCM-012 vision unresolved unit keys: False False cross-reference: False
HCM-013 t

In [3]:
import runpy
module=runpy.run_path('scripts/research/harta_matching_pilot.py')
calibration=Path('reports/generated/source_quality/six_portal_calibration_095214')
result=module['report'](pool,reviews,calibration)
print('Frozen candidate set successfully reproduced.')
print('Location ranking gain:',result['location_score_gain_pairs'])
print('Structured claim conflict pairs:',result['claim_conflict_pairs'])
print('Maximum shared description run:',result['maximum_description_token_overlap'])
assert result['field_enrichment_permitted'] is False

Frozen candidate set successfully reproduced.
Location ranking gain: 21
Structured claim conflict pairs: 13
Maximum shared description run: 7


In [4]:
for s in result['six_source_calibration']['historical_processed']:
    print(s['source_name'], 'unique',s['unique_source_listing_count'],'accepted',s['accepted_count'],'provisional score',s['source_quality_score'],'weight',s['proposed_weight'])
    assert s['proposed_weight']==1 and s['location_score'] is None and s['historical_reliability_score'] is None
assert {s['source_name'] for s in result['six_source_calibration']['historical_processed']}==set(module['SIX'])
harta=json.loads((root/'harta_quality/source_quality_report.json').read_text())
overall=next(s for s in harta['sources'] if s['segment']=='all')
print('Separate Harta assessment:',overall['unique_source_listing_count'],overall['accepted_count'],overall['source_quality_score'],overall['proposed_weight'])
assert overall['source_quality_score']==48.815 and overall['proposed_weight']==1

gjirafa unique 4594 accepted 4315 provisional score 75.517867 weight 1.0
merrjep unique 17572 accepted 14747 provisional score 68.39248 weight 1.0
myrealestate unique 160 accepted 143 provisional score 70.32309 weight 1.0
pro-rks unique 4271 accepted 3214 provisional score 69.275033 weight 1.0
topia unique 516 accepted 513 provisional score 75.622093 weight 1.0
vision unique 113 accepted 98 provisional score 68.927974 weight 1.0
Separate Harta assessment: 50 3 48.815 1.0


In [5]:
audit=json.loads((root/'preservation_audit.json').read_text())
assert audit['database_counts_and_high_waters_unchanged']
assert audit['read_only']=='on' and audit['changed_preexisting_files']==[]
print('DB counts and high waters unchanged:',audit['database_counts_and_high_waters_unchanged'])
print('Pre-existing files preserved:',audit['preexisting_files_checked'])
from groundtruth.portals.registry import get_portal
assert not get_portal('harta').public and not get_portal('harta').weekly
print('Harta public/weekly inclusion remains disabled.')

DB counts and high waters unchanged: True
Pre-existing files preserved: 360
Harta public/weekly inclusion remains disabled.
